## (Optional) Mounting Google Drive
(Comment out in local environment)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# %cd "/content/drive/MyDrive/menutracker"

/content/drive/MyDrive/menutracker


## (Optional) Create and activate virtual environment as needed

In [ ]:
python -m venv venv

# To activate the virtual environment, run the following in your terminal:

# source venv/bin/activate // On Unix or MacOS
# venv\Scripts\activate // On Windows

## Install requirements (TBC)

In [1]:
%pip install -r requirements.txt
# %pip install selenium
# %pip install fake_useragent
# %pip install Scrapy
# %pip install scrapy-selenium

Note: you may need to restart the kernel to use updated packages.


## Initialise dependencies

In [ ]:
from define_collection_wave import folder, create_collection
from helpers import combo_PDFDownload, combo_PDFDownload_class_name, RunSpider, RunScript, java_PDF, greene_king_download, combo_imgDownload, vue_PDF, PDFDownloader, create_folder

create_collection()

Folder 'Sep_collection_2025' already exists.


## Web scraping core scripts

### 1. McDonald's

In [5]:
RunScript('1_McDonalds')

Root path set to: /Users/anguschiu/dev/menutracker
Successfully scraped 1_McDonalds


### 2. Wetherspoons

In [6]:
RunScript('2_Wetherspoons')

Root path set to: /Users/anguschiu/dev/menutracker
Successfully scraped 2_Wetherspoons


### 3. Costa Coffee

In [7]:
RunScript('3_CostaCoffee')

Root path set to: /Users/anguschiu/dev/menutracker
Festive Spice Latte
Caramel Nutcracker Latte
Caramel Nutcracker Hot Chocolate
Caramel Nutcracker Iced Latte
Gingerbread Iced Latte
Black Forest Frappe
Americano
Cappuccino
Chai Latte
Cortado
Decaf Tea
Earl Grey Tea
Flat White
Gingerbread & Cream Latte
Hot Chocolate
Iced Black Americano
Iced Chai Latte
Iced Flat White
Iced Latte
Iced Mocha
Mango & Passion Fruit Cooler
Mocha
Mocha Cortado
Espresso
Red Summer Berries
Terry's Chocolate Orange® Hot Chocolate
White Hot Chocolate
Chocolate Fudge Frappé
Chocolate Fudge Frappé with Coffee
Salted Caramel Frappé
Salted Caramel Frappé with Coffee
Tropical Mango Bubble Frappé
Coffee Frappé
Latte
Strawberry Drizzle Frappé
Black Forest & Cream Hot Chocolate
Peach Ice Tea
Mellow Mango with Zinc
Spiced Apple with Vitamin B6
Citrus Zing with Vitamin C
English Breakfast Tea
Milk Babyccino
Superfruity Infusion
Iced Cappuccino
None
Mint Tea
Green Tea
Festive Spice Flat White
Festive Spice Iced Latte
Chicke

### 4. Greggs

In [8]:
RunScript('4_Greggs')

Root path set to: /Users/anguschiu/dev/menutracker
Successfully scraped 4_Greggs



### 5. KFC

Historically present their nutritional info in PDF format but recently made it available on their website

In [ ]:
RunSpider('5_KFC', folder)

cmd=scrapy crawl 5_KFC -o /content/drive/MyDrive/menutracker/June_collection_2025/5_KFC_Jun-26-2025/5_KFC_items.csv
root_path=/content/drive/MyDrive/menutracker
finished scraping 5_KFC


In [ ]:
%cd "Scrapy_spiders"
!scrapy crawl 5_KFC -o "Sep_collection_2025/5_KFC_Aug-22-2025/5_KFC_items.csv"
%pwd

/Users/anguschiu/dev/menutracker/Scrapy_spiders
/Users/anguschiu/dev/menutracker/Scrapy_spiders/Scrapy_spiders/spiders/a32_LochFyne.py:20: SyntaxWarning: invalid escape sequence '\('
  kcals = re.findall('\([0-9]+?-?[0-9]+ kcal',item_description)
2025-08-22 20:45:12 [scrapy.utils.log] INFO: Scrapy 2.13.3 started (bot: Scrapy_spiders)
2025-08-22 20:45:12 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.1',
 'libxml2': '2.14.5',
 'cssselect': '1.3.0',
 'parsel': '1.10.0',
 'w3lib': '2.3.1',
 'Twisted': '25.5.0',
 'Python': '3.13.5 (main, Jun 11 2025, 15:36:57) [Clang 17.0.0 '
           '(clang-1700.0.13.3)]',
 'pyOpenSSL': '25.1.0 (OpenSSL 3.5.2 5 Aug 2025)',
 'cryptography': '45.0.6',
 'Platform': 'macOS-15.6-arm64-arm-64bit-Mach-O'}
2025-08-22 20:45:12 [selenium.webdriver.common.selenium_manager] DEBUG: Selenium Manager binary found at: /Users/anguschiu/dev/menutracker/venv/lib/python3.13/site-packages/selenium/webdriver/common/macos/selenium-manager
2025-08-22 20:45:12 [selenium.webd

'/Users/anguschiu/dev/menutracker/Scrapy_spiders'

In [ ]:
import scrapy
from selenium import webdriver
from selenium.webdriver.chrome.options import Options # Import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from fake_useragent import UserAgent

class A5KfcSpider(scrapy.Spider):
    name = "a5_KFC"
    allowed_domains = ["www.kfc.co.uk"]
    start_urls = ["https://www.kfc.co.uk/our-food/nutrition"]

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Configure Chrome options for headless mode and no-sandbox
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")

        # Initialize Chrome WebDriver with options. Selenium Manager handles the driver path.
        # Removed executable_path and Service as they are not needed with Selenium Manager
        self.driver = webdriver.Chrome(options=chrome_options)
        self.user_agent = UserAgent()


    def parse(self, response):
        # Use Selenium to interact with the page
        self.driver.get(response.url)

        try:
            # Wait for the elements containing food items to load
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.c-product-card_body__nameWrapper"))
            )

            # Get the page source after interactions
            page_source = self.driver.page_source

            # Now use Scrapy selectors on the page source
            selector = scrapy.Selector(text=page_source)

            # Extract food item names
            food_items = selector.css('div.c-product-card_body__nameWrapper::text').getall()

            # You can add more scraping logic here to extract other details like nutrition information

            for item_name in food_items:
                yield {
                    'item_name': item_name.strip()
                    # Add other extracted data here
                }

        finally:
            # Close the browser
            self.driver.quit()